In [1]:
import subprocess, sys
from pathlib import Path

# Install ONNX runtime
ONNX_WHL = "/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--no-deps", ONNX_WHL], check=True)

import onnxruntime as ort
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import soundfile as sf
import re, gc, time
from scipy.ndimage import gaussian_filter1d

print("Imports done.")

# Verify all key files exist
checks = {
    "Perch ONNX":    "/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx",
    "SED fold0":     "/kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public/sed_fold0.onnx",
    "TaxaMoE model": "/kaggle/input/notebooks/kritikabenjwal/birdclef26-taxamoe-training/taxamoe_model_v3.pt",
}

for name, path in checks.items():
    p = Path(path)
    if p.exists():
        print(f"  ✓ {name}: {p.stat().st_size/1e6:.1f} MB")
    else:
        print(f"  ✗ {name}: NOT FOUND — {path}")

Imports done.
  ✓ Perch ONNX: 409.1 MB
  ✓ SED fold0: 19.7 MB
  ✓ TaxaMoE model: 12.3 MB


In [2]:
for p in Path('/kaggle/input').rglob('taxamoe_model*.pt'):
    print(f"{p.stat().st_size/1e6:.1f} MB  {p}")

12.3 MB  /kaggle/input/notebooks/kritikabenjwal/birdclef26-taxamoe-training/taxamoe_model_v3.pt
11.6 MB  /kaggle/input/notebooks/kritikabenjwal/birdclef26-taxamoe-training/taxamoe_model.pt


In [3]:
# Explicitly load v3
TAXAMOE_PATH = Path('/kaggle/input/notebooks/kritikabenjwal/birdclef26-taxamoe-training/taxamoe_model_v3.pt')

if TAXAMOE_PATH.exists():
    print(f"TaxaMoE v3 found: {TAXAMOE_PATH}")
    print(f"Size: {TAXAMOE_PATH.stat().st_size/1e6:.1f} MB")
else:
    print("NOT FOUND")

TaxaMoE v3 found: /kaggle/input/notebooks/kritikabenjwal/birdclef26-taxamoe-training/taxamoe_model_v3.pt
Size: 12.3 MB


In [4]:
BASE       = Path('/kaggle/input/competitions/birdclef-2026')
PERCH_PATH = "/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx"
SED_DIR    = Path("/kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public")

# Find TaxaMoE model
TAXAMOE_PATH = None
for p in Path('/kaggle/input').rglob('taxamoe_model_v3.pt'):
    TAXAMOE_PATH = p
    print(f"TaxaMoE v3 found: {p}")
SR             = 32000
N_WINDOWS      = 12
WINDOW_SAMPLES = SR * 5
FILE_SAMPLES   = SR * 60

taxonomy   = pd.read_csv(BASE / 'taxonomy.csv')
sample_sub = pd.read_csv(BASE / 'sample_submission.csv')

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}

FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m: return {"site": "unknown", "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}

def get_taxa_group(class_name):
    return {'Aves':0,'Insecta':1,'Amphibia':2,'Mammalia':3,'Reptilia':4}.get(class_name,-1)

taxonomy['taxa_group'] = taxonomy['class_name'].map(get_taxa_group)
# Add this after taxonomy['taxa_group'] = ... line
LABELS_PATH = "/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/labels.csv"
bc_labels   = pd.read_csv(LABELS_PATH).reset_index()
bc_labels.columns = ['bc_index', 'scientific_name']
NO_LABEL    = len(bc_labels)

mapping  = taxonomy.merge(bc_labels, on='scientific_name', how='left')
mapping['bc_index'] = mapping['bc_index'].fillna(NO_LABEL).astype(int)
lbl2bc   = mapping.set_index('primary_label')['bc_index']

BC_INDICES  = [int(lbl2bc.loc[c]) if c in lbl2bc.index else NO_LABEL for c in PRIMARY_LABELS]
MAPPED_MASK = [idx != NO_LABEL for idx in BC_INDICES]
MAPPED_POS  = [i for i, m in enumerate(MAPPED_MASK) if m]
MAPPED_BC   = [BC_INDICES[i] for i in MAPPED_POS]
print(f"Classes:  {N_CLASSES}")
print(f"Taxonomy: {len(taxonomy)} species")

# Test paths
test_paths = sorted((BASE / 'test_soundscapes').glob('*.ogg'))
if not test_paths:
    print("No hidden test — using train soundscapes for dry run")
    test_paths = sorted((BASE / 'train_soundscapes').glob('*.ogg'))[:20]
else:
    print(f"Hidden test files: {len(test_paths)}")

TaxaMoE v3 found: /kaggle/input/notebooks/kritikabenjwal/birdclef26-taxamoe-training/taxamoe_model_v3.pt
Classes:  234
Taxonomy: 234 species
No hidden test — using train soundscapes for dry run


In [5]:
import numpy as np
import torch.serialization

# PyTorch 2.6 fix
torch.serialization.add_safe_globals([np.dtype])

class CrossTaxaContrastiveAdapter(nn.Module):
    def __init__(self, input_dim=1536, output_dim=512, hidden_dim=768, dropout=0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim), nn.LayerNorm(output_dim))
    def forward(self, x):
        return F.normalize(self.encoder(x), dim=-1)

class AvesHead(nn.Module):
    def __init__(self, input_dim=512, n_species=162, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.LayerNorm(256),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(256, n_species))
    def forward(self, x): return self.net(x)

class InsectaHead(nn.Module):
    def __init__(self, input_dim=512, n_species=28, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.LayerNorm(128),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(128, n_species))
    def forward(self, x): return self.net(x)

class PrototypicalHead(nn.Module):
    def __init__(self, input_dim=512, n_species=35, dropout=0.2):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, 256), nn.LayerNorm(256),
            nn.GELU(), nn.Dropout(dropout))
        self.prototypes  = nn.Parameter(torch.randn(n_species, 256))
        self.temperature = nn.Parameter(torch.tensor(10.0))
    def forward(self, x):
        h = F.normalize(self.proj(x), dim=-1)
        p = F.normalize(self.prototypes, dim=-1)
        return torch.matmul(h, p.T) * F.softplus(self.temperature)

class TaxaMoE(nn.Module):
    def __init__(self, taxonomy_df, input_dim=1536):
        super().__init__()
        self.n_aves     = (taxonomy_df['taxa_group']==0).sum()
        self.n_insecta  = (taxonomy_df['taxa_group']==1).sum()
        self.n_amphibia = (taxonomy_df['taxa_group']==2).sum()
        self.n_mammalia = (taxonomy_df['taxa_group']==3).sum()
        self.n_reptilia = (taxonomy_df['taxa_group']==4).sum()
        self.n_classes  = len(taxonomy_df)
        self.register_buffer('aves_idx',
            torch.tensor(taxonomy_df[taxonomy_df['taxa_group']==0].index.tolist()))
        self.register_buffer('insecta_idx',
            torch.tensor(taxonomy_df[taxonomy_df['taxa_group']==1].index.tolist()))
        self.register_buffer('amphibia_idx',
            torch.tensor(taxonomy_df[taxonomy_df['taxa_group']==2].index.tolist()))
        self.register_buffer('mammalia_idx',
            torch.tensor(taxonomy_df[taxonomy_df['taxa_group']==3].index.tolist()))
        self.register_buffer('reptilia_idx',
            torch.tensor(taxonomy_df[taxonomy_df['taxa_group']==4].index.tolist()))
        self.adapter      = CrossTaxaContrastiveAdapter(input_dim, 512)
        self.router       = nn.Sequential(
            nn.Linear(512,128), nn.GELU(), nn.Linear(128,5))
        self.aves_head     = AvesHead(512, self.n_aves)
        self.insecta_head  = InsectaHead(512, self.n_insecta)
        self.amphibia_head = PrototypicalHead(512, self.n_amphibia)
        self.mammalia_head = PrototypicalHead(512, self.n_mammalia)
        self.reptilia_head = PrototypicalHead(512, self.n_reptilia)

    def forward(self, x):
        z  = self.adapter(x)
        rw = F.softmax(self.router(z), dim=-1)
        batch  = x.shape[0]
        logits = torch.zeros(batch, self.n_classes, device=x.device)
        logits[:, self.aves_idx]     = self.aves_head(z)     * rw[:, 0:1]
        logits[:, self.insecta_idx]  = self.insecta_head(z)  * rw[:, 1:2]
        logits[:, self.amphibia_idx] = self.amphibia_head(z) * rw[:, 2:3]
        logits[:, self.mammalia_idx] = self.mammalia_head(z) * rw[:, 3:4]
        logits[:, self.reptilia_idx] = self.reptilia_head(z) * rw[:, 4:5]
        return logits

# Load trained weights
taxonomy_indexed = taxonomy.reset_index(drop=True)
taxamoe = TaxaMoE(taxonomy_indexed, input_dim=1770)
checkpoint = torch.load(TAXAMOE_PATH, map_location='cpu', weights_only=False)
taxamoe.load_state_dict(checkpoint['model_state'])
taxamoe.eval()

print(f"TaxaMoE loaded. Train AUC was: {checkpoint['auc']:.4f}")
dummy = torch.randn(4, 1770)
with torch.no_grad():
    out = taxamoe(dummy)
print(f"Output shape: {out.shape}")
print("TaxaMoE ready for inference.")

TaxaMoE loaded. Train AUC was: 0.9987
Output shape: torch.Size([4, 234])
TaxaMoE ready for inference.


In [6]:
sess = ort.InferenceSession(PERCH_PATH, providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name
out_names  = {o.name: i for i, o in enumerate(sess.get_outputs())}

def run_perch_batch(paths, batch_size=8):
    all_emb  = []
    all_logits = [] 
    all_meta = []

    for i in range(0, len(paths), batch_size):
        batch = paths[i:i+batch_size]
        x = np.zeros((len(batch)*N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)

        for bi, path in enumerate(batch):
            y, _ = sf.read(str(path), dtype='float32', always_2d=False)
            if y.ndim == 2: y = y.mean(1)
            if len(y) < FILE_SAMPLES: y = np.pad(y,(0,FILE_SAMPLES-len(y)))
            else: y = y[:FILE_SAMPLES]
            x[bi*N_WINDOWS:(bi+1)*N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)

            meta = parse_fname(Path(path).name)
            stem = Path(path).stem
            for w in range(N_WINDOWS):
                end_t = (w+1)*5
                all_meta.append({
                    'row_id':   f"{stem}_{end_t}",
                    'filename': Path(path).name,
                    'site':     meta['site'],
                    'hour_utc': meta['hour_utc']
                })

        outs = sess.run(None, {input_name: x})
        all_emb.append(outs[out_names['embedding']])
        all_logits.append(outs[out_names['label']]) 
        gc.collect()

    return np.vstack(all_emb).astype(np.float32),np.vstack(all_logits).astype(np.float32),  pd.DataFrame(all_meta)

print("Running Perch on test files...")
t0 = time.time()
emb_te, logits_te, meta_te = run_perch_batch(test_paths)
perch_logits_te = np.zeros((len(emb_te), N_CLASSES), dtype=np.float32)
perch_logits_te[:, MAPPED_POS] = logits_te[:, MAPPED_BC]
emb_te_full = np.hstack([emb_te, perch_logits_te])   # (N, 1770)
print(f"Full input shape: {emb_te_full.shape}")
print(f"Done in {time.time()-t0:.1f}s")
print(f"Embeddings: {emb_te.shape}")
print(f"Meta rows:  {len(meta_te)}")

Running Perch on test files...
Full input shape: (240, 1770)
Done in 53.9s
Embeddings: (240, 1536)
Meta rows:  240


In [7]:
# ── Bayesian site×hour prior (v40 — proven to work) ──────────────────
print("Building Bayesian site×hour prior...")

sc_label_df = pd.read_csv(BASE / 'train_soundscapes_labels.csv')
sc_label_df['end_sec'] = pd.to_timedelta(
    sc_label_df['end']).dt.total_seconds().astype(int)

def parse_meta(fname):
    m = FNAME_RE.match(fname)
    if not m: return 'unknown', -1
    _, site, _, hms = m.groups()
    return site, int(hms[:2])

sc_label_df['site'] = sc_label_df['filename'].apply(lambda x: parse_meta(x)[0])
sc_label_df['hour'] = sc_label_df['filename'].apply(lambda x: parse_meta(x)[1])
sc_label_df['row_id'] = (sc_label_df['filename'].str.replace('.ogg','',regex=False)
                         + '_' + sc_label_df['end_sec'].astype(str))

global_counts = np.zeros(N_CLASSES, dtype=np.float64)
sh_counts     = {}
sh_windows    = {}
total_windows = 0

for _, row in sc_label_df.iterrows():
    key = (row['site'], row['hour'])
    if key not in sh_counts:
        sh_counts[key]  = np.zeros(N_CLASSES, dtype=np.float64)
        sh_windows[key] = 0
    for lbl in str(row['primary_label']).split(';'):
        lbl = lbl.strip()
        if lbl in label_to_idx:
            idx = label_to_idx[lbl]
            global_counts[idx]  += 1
            sh_counts[key][idx] += 1
    sh_windows[key] += 1
    total_windows   += 1

SHRINKAGE    = 8
global_prior = global_counts / (total_windows + 1e-8)

def get_prior(site, hour):
    key = (site, hour)
    if key in sh_counts:
        n     = sh_windows[key]
        w     = n / (n + SHRINKAGE)
        local = sh_counts[key] / (n + 1e-8)
        return np.clip(w * local + (1-w) * global_prior, 1e-4, 1-1e-4).astype(np.float32)
    return np.clip(global_prior, 1e-4, 1-1e-4).astype(np.float32)

# Store as dicts for Cell 9 lookup
file_sites = meta_te.groupby('filename')['site'].first().to_dict()
file_hours = meta_te.groupby('filename')['hour_utc'].first().to_dict()

print(f"Prior built: {len(sh_counts)} (site,hour) pairs")
print(f"Global prior nonzero: {(global_prior > 0.001).sum()} species")

Building Bayesian site×hour prior...
Prior built: 25 (site,hour) pairs
Global prior nonzero: 75 species


In [8]:
# ── Post-processing helper functions ─────────────────────────────────
import pandas as pd
import numpy as np
from scipy.ndimage import gaussian_filter1d

def rank_aware_scaling(probs, n_windows=12, power=0.70):
    N, C = probs.shape
    view     = probs.reshape(-1, n_windows, C)
    file_max = view.max(axis=1, keepdims=True)
    scaled   = view * np.power(np.clip(file_max, 1e-8, 1.0), power)
    return scaled.reshape(N, C)

def file_confidence_scale(probs, n_windows=12, top_k=2, power=0.40):
    N, C = probs.shape
    view       = probs.reshape(-1, n_windows, C)
    sorted_v   = np.sort(view, axis=1)
    top_k_mean = sorted_v[:, -top_k:, :].mean(axis=1, keepdims=True)
    return (view * np.power(np.clip(top_k_mean, 1e-8, 1.0), power)).reshape(N, C)

def adaptive_delta_smooth(probs, n_windows=12, base_alpha=0.20):
    """
    Confidence-weighted temporal smoothing — applied to PROBABILITY values only.
    High-confidence windows (conf≈1) → alpha≈0 (untouched).
    Low-confidence windows (conf≈0)  → alpha≈base_alpha (smoothed toward neighbours).
    Must be called on probability arrays, NOT on rank percentile arrays.
    base_alpha=0.20 matches the 0.949 notebook.
    """
    N, C   = probs.shape
    result = probs.copy()
    view   = probs.reshape(-1, n_windows, C)
    out    = result.reshape(-1, n_windows, C)
    for t in range(n_windows):
        conf  = view[:, t, :].max(axis=-1, keepdims=True)
        alpha = base_alpha * (1.0 - conf)
        if   t == 0:             nbr = (view[:, t,   :] + view[:, t+1, :]) / 2.0
        elif t == n_windows - 1: nbr = (view[:, t-1, :] + view[:, t,   :]) / 2.0
        else:                    nbr = (view[:, t-1, :] + view[:, t+1, :]) / 2.0
        out[:, t, :] = (1.0 - alpha) * view[:, t, :] + alpha * nbr
    return result

print("Helper functions defined (rank_aware_scaling, file_confidence_scale, adaptive_delta_smooth).")

Helper functions defined (rank_aware_scaling, file_confidence_scale, adaptive_delta_smooth).


In [9]:
# ── Inline BiGRU ProtoSSM — trained on perch-meta cached embeddings ──
import torch, torch.nn as nn, torch.nn.functional as F, time as _t

lproto_probs = None

try:
    _npz  = list(Path('/kaggle/input').rglob('full_perch_arrays.npz'))[0]
    _meta = list(Path('/kaggle/input').rglob('full_perch_meta.parquet'))[0]
    _arr  = np.load(_npz, allow_pickle=True)
    print("NPZ keys:", list(_arr.files))
    for k in _arr.files:
        print(f"  {k}: shape={_arr[k].shape}, dtype={_arr[k].dtype}")

    meta_tr = pd.read_parquet(_meta)
    print(f"meta_tr shape: {meta_tr.shape}, columns: {meta_tr.columns.tolist()}")
    print(f"meta_tr head:\n{meta_tr.head(3)}")

    # ── Extract embeddings (1536-dim) ─────────────────────────────────
    emb_tr_raw = None
    for k in _arr.files:
        a = _arr[k]
        if a.ndim == 2 and a.shape[1] == 1536:
            emb_tr_raw = a.astype(np.float32)
            print(f"Found embeddings in key '{k}': {emb_tr_raw.shape}")
            break
    if emb_tr_raw is None:
        raise KeyError(f"No 1536-dim array found. Keys/shapes: {[(k, _arr[k].shape) for k in _arr.files]}")

    # ── Extract score signal (scores_full_raw: 234-dim ProtoSSM scores) ──
    if 'scores_full_raw' in _arr.files:
        sc_tr_raw = _arr['scores_full_raw'].astype(np.float32)  # (708, 234)
        print(f"Found scores_full_raw: {sc_tr_raw.shape}")
    else:
        sc_tr_raw = None
        print("No scores_full_raw found — will train on embeddings only")

    # ── Build row_id if missing ────────────────────────────────────────
    if 'row_id' not in meta_tr.columns:
        n_wins = N_WINDOWS
        n_files_tr = len(emb_tr_raw) // n_wins
        rids = []
        fnames_unique = meta_tr['filename'].unique() if 'filename' in meta_tr.columns else [f"file_{i}" for i in range(n_files_tr)]
        for fn in fnames_unique:
            stem = str(fn).replace('.ogg', '')
            for w in range(n_wins):
                rids.append(f"{stem}_{(w+1)*5}")
        meta_tr['row_id'] = rids[:len(meta_tr)]

    # ── Build label matrix from train_soundscapes_labels.csv ──────────
    _labels = pd.read_csv(BASE / 'train_soundscapes_labels.csv')
    _labels['end_sec'] = pd.to_timedelta(_labels['end']).dt.total_seconds().astype(int)
    _labels['row_id']  = (_labels['filename'].str.replace('.ogg','',regex=False)
                          + '_' + _labels['end_sec'].astype(str))

    Y_tr = np.zeros((len(meta_tr), N_CLASSES), dtype=np.float32)
    _r2p = {r: i for i, r in enumerate(meta_tr['row_id'])}
    matched = 0
    for _, _r in _labels.iterrows():
        pos = _r2p.get(str(_r['row_id']))
        if pos is None:
            continue
        for lbl in str(_r['primary_label']).split(';'):
            lbl = lbl.strip()
            if lbl in label_to_idx:
                Y_tr[pos, label_to_idx[lbl]] = 1.0
                matched += 1
    print(f"Label matrix: {Y_tr.shape}, matched windows: {matched}, "
          f"active classes: {(Y_tr.sum(0)>0).sum()}")

    if (Y_tr.sum(0) > 0).sum() < 10:
        raise ValueError(f"Too few active classes ({(Y_tr.sum(0)>0).sum()}) — row_id mismatch. "
                         f"Sample meta_tr row_ids: {meta_tr['row_id'].iloc[:3].tolist()}\n"
                         f"Sample label row_ids: {_labels['row_id'].iloc[:3].tolist()}")

    # ── Reshape to (n_files, N_WINDOWS, dim) ──────────────────────────
    n_tr = len(emb_tr_raw) // N_WINDOWS
    E  = emb_tr_raw[:n_tr * N_WINDOWS].reshape(n_tr, N_WINDOWS, 1536)
    Yf = Y_tr[:n_tr * N_WINDOWS].reshape(n_tr, N_WINDOWS, N_CLASSES)

    if sc_tr_raw is not None:
        Sf = sc_tr_raw[:n_tr * N_WINDOWS].reshape(n_tr, N_WINDOWS, N_CLASSES)
        print(f"Score residual signal shape: {Sf.shape}")
    else:
        Sf = np.zeros((n_tr, N_WINDOWS, N_CLASSES), dtype=np.float32)
        print("Using zero score signal")

    print(f"Training shapes — E:{E.shape}, Sf:{Sf.shape}, Yf:{Yf.shape}")

    # ── BiGRU architecture with score residual ────────────────────────
    class BiGRUProto(nn.Module):
        def __init__(self, d_in=1536, d_sc=234, d_h=128, n_cls=234, drop=0.20):
            super().__init__()
            self.proj = nn.Sequential(
                nn.Linear(d_in, d_h), nn.LayerNorm(d_h), nn.GELU(), nn.Dropout(drop))
            self.gru  = nn.GRU(d_h, d_h, num_layers=2,
                               batch_first=True, bidirectional=True, dropout=drop)
            self.norm = nn.LayerNorm(d_h * 2)
            self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(d_h*2, n_cls))
            # Learnable blend weight for score residual
            self.sc_alpha = nn.Parameter(torch.tensor(0.15))
        def forward(self, x, sc=None):
            h = self.proj(x)
            h, _ = self.gru(h)
            h = self.norm(h)
            out = self.head(h)
            if sc is not None:
                out = out + torch.sigmoid(self.sc_alpha) * sc
            return out

    E_t  = torch.tensor(E,  dtype=torch.float32)
    S_t  = torch.tensor(Sf, dtype=torch.float32)
    Y_t  = torch.tensor(Yf, dtype=torch.float32)

    pos_cnt   = Y_t.sum(dim=(0, 1))
    total_win = Y_t.shape[0] * Y_t.shape[1]
    pos_w     = ((total_win - pos_cnt + 1) / (pos_cnt + 1)).clamp(max=20.0)

    model = BiGRUProto(n_cls=N_CLASSES)
    opt   = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

    best_loss, best_state = float('inf'), None
    t0_tr = _t.time()
    print("Training BiGRU ProtoSSM with score residual (50 epochs)...")

    for ep in range(50):
        model.train()
        out  = model(E_t, S_t)
        loss = F.binary_cross_entropy_with_logits(
            out, Y_t, pos_weight=pos_w[None, None, :])
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
        if loss.item() < best_loss:
            best_loss  = loss.item()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if (ep + 1) % 10 == 0:
            print(f"  Epoch {ep+1}: loss={loss.item():.4f}")

    model.load_state_dict(best_state)
    model.eval()
    print(f"Training done in {_t.time()-t0_tr:.0f}s. Best loss: {best_loss:.4f}")

    # ── Inference on test with TTA (forward + temporal flip) ──────────
    n_te  = len(test_paths)
    E_te  = torch.tensor(
        emb_te[:n_te * N_WINDOWS].reshape(n_te, N_WINDOWS, 1536),
        dtype=torch.float32)
    # Use perch_logits_te as score signal for test inference
    S_te  = torch.tensor(
        perch_logits_te[:n_te * N_WINDOWS].reshape(n_te, N_WINDOWS, N_CLASSES),
        dtype=torch.float32)

    with torch.no_grad():
        _logits_fwd = model(E_te, S_te).numpy()
        _logits_rev = model(E_te.flip(1), S_te.flip(1)).numpy()[:, ::-1, :].copy()

    _logits_avg  = (_logits_fwd + _logits_rev) / 2.0
    lproto_probs = (1 / (1 + np.exp(-np.clip(_logits_avg, -30, 30)))).astype(np.float32)
    lproto_probs = gaussian_filter1d(lproto_probs, sigma=0.65, axis=1, mode='nearest')
    lproto_probs = lproto_probs.reshape(-1, N_CLASSES)
    print(f"BiGRU probs: {lproto_probs.shape}, "
          f"range [{lproto_probs.min():.4f}, {lproto_probs.max():.4f}], "
          f"nonzero cols: {(lproto_probs.mean(0) > 0.001).sum()}")

    del E_t, S_t, Y_t, emb_tr_raw
    import gc; gc.collect()

except Exception as e:
    import traceback
    print(f"BiGRU training failed: {e}")
    traceback.print_exc()
    lproto_probs = None
    print("Falling back to 3-component blend (v40).")

NPZ keys: ['scores_full_raw', 'emb_full']
  scores_full_raw: shape=(708, 234), dtype=float32
  emb_full: shape=(708, 1536), dtype=float32
meta_tr shape: (708, 4), columns: ['row_id', 'filename', 'site', 'hour_utc']
meta_tr head:
                                     row_id  \
0   BC2026_Train_0001_S08_20250606_030007_5   
1  BC2026_Train_0001_S08_20250606_030007_10   
2  BC2026_Train_0001_S08_20250606_030007_15   

                                    filename site  hour_utc  
0  BC2026_Train_0001_S08_20250606_030007.ogg  S08         3  
1  BC2026_Train_0001_S08_20250606_030007.ogg  S08         3  
2  BC2026_Train_0001_S08_20250606_030007.ogg  S08         3  
Found embeddings in key 'emb_full': (708, 1536)
Found scores_full_raw: (708, 234)
Label matrix: (708, 234), matched windows: 6174, active classes: 71
Score residual signal shape: (59, 12, 234)
Training shapes — E:(59, 12, 1536), Sf:(59, 12, 234), Yf:(59, 12, 234)
Training BiGRU ProtoSSM with score residual (50 epochs)...
  Epoch 10:

In [10]:
import librosa
import time
t0 = time.time()

# ── Step 1: TaxaMoE predictions ──────────────────────────────────────
print("Running TaxaMoE inference...")
taxamoe.eval()
taxamoe_preds = []
with torch.no_grad():
    for i in range(0, len(emb_te_full), 128):
        batch  = torch.tensor(emb_te_full[i:i+128], dtype=torch.float32)
        logits = taxamoe(batch)
        probs  = torch.sigmoid(logits).numpy()
        taxamoe_preds.append(probs)
taxamoe_probs = np.vstack(taxamoe_preds).astype(np.float32)
print(f"TaxaMoE probs: {taxamoe_probs.shape}")

# ── Step 1b: ProtoSSM v4 predictions ─────────────────────────────────
print("\nRunning ProtoSSM v4 inference...")
PROTOSSM_PATH = "/kaggle/input/notebooks/dingjiarun/pantanal-distill-birdclef2026-onnx/protossm_v4.onnx"
proto_sess    = ort.InferenceSession(PROTOSSM_PATH, providers=["CPUExecutionProvider"])

n_files  = len(test_paths)
site_ids = meta_te.groupby('filename')['site'].first().values

def site_to_int(s):
    try: return int(str(s).replace('S','').lstrip('0') or '0')
    except: return 0

site_arr = np.array([min(site_to_int(s), 19) for s in site_ids], dtype=np.int64)
hour_arr = meta_te.groupby('filename')['hour_utc'].first().values.astype(np.int64)

emb_reshaped    = emb_te.reshape(n_files, N_WINDOWS, 1536)
logits_reshaped = perch_logits_te.reshape(n_files, N_WINDOWS, N_CLASSES)

proto_preds = []
for i in range(n_files):
    out = proto_sess.run(None, {
        'emb':    emb_reshaped[i:i+1].astype(np.float32),
        'logits': logits_reshaped[i:i+1].astype(np.float32),
        'site':   site_arr[i:i+1],
        'hour':   hour_arr[i:i+1],
    })
    proto_preds.append(out[0].squeeze(0))

proto_probs = np.vstack(proto_preds).astype(np.float32)
proto_probs = 1 / (1 + np.exp(-proto_probs))
print(f"ProtoSSM v4 probs: {proto_probs.shape}")

# ── Apply Bayesian prior in LOGIT SPACE (lambda=0.625 — v40 proven value) ─
LAMBDA_PRIOR = 0.625

if 'get_prior' in vars() or 'get_prior' in globals():
    print(f"Applying logit-space Bayesian prior (lambda={LAMBDA_PRIOR})...")
    prior_p = np.ones((len(meta_te), N_CLASSES), dtype=np.float32)
    for i, fname in enumerate(meta_te['filename'].values):
        site = file_sites.get(fname, list(file_sites.values())[0])
        hour = file_hours.get(fname, -1)
        prior_p[i] = get_prior(site, hour)
    prior_p = np.clip(prior_p, 1e-4, 1.0 - 1e-4)

    proto_logits    = np.log(np.clip(proto_probs, 1e-7, 1.0) /
                             np.clip(1.0 - proto_probs, 1e-7, 1.0))
    prior_logodds   = np.log(prior_p) - np.log(1.0 - prior_p)
    proto_probs_adj = (1.0 / (1.0 + np.exp(
                        -(proto_logits + LAMBDA_PRIOR * prior_logodds)
                      ))).astype(np.float32)
    print(f"  Proto adj range: [{proto_probs_adj.min():.4f}, {proto_probs_adj.max():.4f}]")
else:
    proto_probs_adj = proto_probs

# ── rank_aware_scaling + file_confidence_scale on proto_probs_adj ─────
# Applied on probability values BEFORE rank conversion — correct order.
# DO NOT apply adaptive_delta_smooth here — it caused regression in v42/v43/v44.
proto_probs_adj = rank_aware_scaling(proto_probs_adj,    N_WINDOWS, power=0.70)
proto_probs_adj = file_confidence_scale(proto_probs_adj, N_WINDOWS, top_k=2, power=0.40)
proto_probs_adj = np.clip(proto_probs_adj, 0, 1)
print(f"Proto probs after scaling: [{proto_probs_adj.min():.4f}, {proto_probs_adj.max():.4f}]")

# ── Step 1c: Perch direct logits ──────────────────────────────────────
perch_direct = 1 / (1 + np.exp(-np.clip(perch_logits_te, -10, 10)))
print(f"Perch direct: {perch_direct.shape}, nonzero: {(perch_direct.mean(0) > 0.01).sum()}")

# ── Step 2: SED inference ─────────────────────────────────────────────
print("\nRunning SED inference...")
def make_sed_session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 4
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), sess_options=so,
                                providers=["CPUExecutionProvider"])

sed_paths    = sorted(SED_DIR.glob("sed_fold*.onnx"))
sed_sessions = [make_sed_session(p) for p in sed_paths]
print(f"Loaded {len(sed_sessions)} SED folds")

N_MELS=256; N_FFT=2048; HOP=512; FMIN=20; FMAX=16000; TOP_DB=80

def audio_to_mel(chunks):
    mels = []
    for x in chunks:
        s = librosa.feature.melspectrogram(
            y=x, sr=SR, n_fft=N_FFT, hop_length=HOP,
            n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0)
        s = librosa.power_to_db(s, top_db=TOP_DB)
        s = (s - s.mean()) / (s.std() + 1e-6)
        mels.append(s)
    return np.stack(mels)[:, None].astype(np.float32)

sed_rows, sed_preds = [], []
for path in test_paths:
    y, _ = sf.read(str(path), dtype='float32', always_2d=False)
    if y.ndim == 2: y = y.mean(1)
    n = FILE_SAMPLES
    if len(y) < n: y = np.pad(y, (0, n-len(y)))
    else: y = y[:n]
    chunks = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
    mel    = audio_to_mel(chunks)
    p_sum  = np.zeros((N_WINDOWS, N_CLASSES), dtype=np.float32)
    for sess_sed in sed_sessions:
        outs        = sess_sed.run(None, {sess_sed.get_inputs()[0].name: mel})
        clip_logits = outs[0]
        frame_max   = outs[1].max(axis=1)
        p_sum += 0.5*(1/(1+np.exp(-clip_logits))) + 0.5*(1/(1+np.exp(-frame_max)))
    p_mean = p_sum / len(sed_sessions)
    p_mean = gaussian_filter1d(p_mean, sigma=0.65, axis=0, mode='nearest')
    sed_preds.append(p_mean)
    stem = Path(path).stem
    sed_rows.extend([f"{stem}_{(w+1)*5}" for w in range(N_WINDOWS)])

sed_probs = np.vstack(sed_preds).astype(np.float32)
print(f"SED probs: {sed_probs.shape}")

# ── Step 3: Rank blend ────────────────────────────────────────────────
print("\nBlending...")
EPS = 1e-5

rb_raw = pd.DataFrame(np.clip(sed_probs,       EPS, 1-EPS)).rank(axis=0, pct=True).to_numpy(np.float32)
rc_raw = pd.DataFrame(np.clip(proto_probs_adj, EPS, 1-EPS)).rank(axis=0, pct=True).to_numpy(np.float32)
rd_raw = pd.DataFrame(np.clip(perch_direct,    EPS, 1-EPS)).rank(axis=0, pct=True).to_numpy(np.float32)

POWER = 1.2
rb = np.power(rb_raw, POWER)
rc = np.power(rc_raw, POWER)
rd = np.power(rd_raw, POWER)

# ── BiGRU guard: only use if it trained AND produced valid output ──────
_use_bigru = (
    'lproto_probs' in dir() and
    lproto_probs is not None and
    lproto_probs.shape == sed_probs.shape and
    np.isfinite(lproto_probs).all() and
    lproto_probs.max() > 0.001
)

if _use_bigru:
    re_raw = pd.DataFrame(np.clip(lproto_probs, EPS, 1-EPS)).rank(
                 axis=0, pct=True).to_numpy(np.float32)
    re = np.power(re_raw, POWER)
    # BiGRU gets 20%, ProtoSSM v4 gets 30% (ProtoSSM is stronger than BiGRU)
    blended = 0.42 * rb + 0.30 * rc + 0.08 * rd + 0.20 * re
    print("4-component blend: SED 42% / ProtoV4 30% / Perch 8% / BiGRU 20%")
else:
    # v40 exact fallback — 0.944 guaranteed
    blended = 0.52 * rb + 0.37 * rc + 0.11 * rd
    print("3-component blend (v40 exact: SED 52% / ProtoV4 37% / Perch 11%)")

# ── Gate 1: Noise suppression ─────────────────────────────────────────
p_sed      = np.clip(sed_probs, EPS, 1-EPS)
rank_proto = pd.DataFrame(proto_probs_adj).rank(axis=0, pct=True).to_numpy(np.float32)
rank_sed   = pd.DataFrame(p_sed).rank(axis=0, pct=True).to_numpy(np.float32)

fake_only = (proto_probs_adj > 0.50) & (p_sed < 0.05)
blended   = np.where(fake_only, 0.92 * blended + 0.08 * rank_proto, blended)

# ── Gate 2: Temporal continuity ───────────────────────────────────────
file_ids     = meta_te['filename'].to_numpy()
offs         = np.arange(-3, 4, dtype=np.float32)
proto_kernel = (1.0 + (offs / 1.20)**2 / 2.0)**(-1.5)
proto_kernel = (proto_kernel / proto_kernel.sum()).astype(np.float32)

pa_ctx = proto_probs_adj.copy()
for fid in pd.unique(file_ids):
    m = file_ids == fid
    x = proto_probs_adj[m]
    if len(x) > 1:
        xp = np.pad(x, ((3,3),(0,0)), mode='edge')
        pa_ctx[m] = sum(proto_kernel[i] * xp[i:i+len(x)] for i in range(7))

xctx       = pd.DataFrame(pa_ctx).rank(axis=0, pct=True).to_numpy(np.float32)
proto_cont = (xctx > 0.88) & (rank_proto > 0.75) & (p_sed < 0.12) & (~fake_only)
blended    = np.where(proto_cont,
                      0.85 * blended + 0.15 * np.maximum(rank_proto, xctx),
                      blended)

# ── Gate 3: SED spike preservation ────────────────────────────────────
sed_only = (rank_sed > 0.95) & (rank_proto < 0.80) & (~fake_only) & (~proto_cont)
blended  = np.where(sed_only, 0.88 * blended + 0.12 * rank_sed, blended)
print("Gates 1-3 applied.")

# ── Gate 4: Sonotype mirroring ────────────────────────────────────────
MIRROR_PAIRS = (
    ("47158son15", "47158son16"),
    ("47158son09", "47158son12"),
    ("47158son02", "47158son14"),
    ("47158son13", "47158son21", "47158son22", "47158son23"),
)
col_to_idx = {label: i for i, label in enumerate(PRIMARY_LABELS)}
mirror_count = 0
for group in MIRROR_PAIRS:
    valid_idx = [col_to_idx[s] for s in group if s in col_to_idx]
    if len(valid_idx) >= 2:
        group_max = blended[:, valid_idx].max(axis=1, keepdims=True)
        blended[:, valid_idx] = group_max
        mirror_count += len(valid_idx)
print(f"Sonotype mirroring: {mirror_count} columns.")

# ── Gate 5: Rare-class suppression ────────────────────────────────────
# Matches 0.949 exactly: only Amphibia, Mammalia, Reptilia (no Insecta)
try:
    tax_df = pd.read_csv(BASE / 'taxonomy.csv').set_index('primary_label')
    rare_classes = {'Amphibia', 'Mammalia', 'Reptilia'}
    rare_count   = 0
    for ci, species in enumerate(PRIMARY_LABELS):
        if species in tax_df.index and \
           tax_df.loc[species, 'class_name'] in rare_classes:
            vals = blended[:, ci]
            thr  = vals.mean() + 0.05
            blended[:, ci] = np.where(vals < thr, vals * 0.90, vals)
            rare_count += 1
    print(f"Rare-class suppression: {rare_count} species.")
except Exception as e:
    print(f"Rare-class suppression skipped: {e}")

# ── Step 3c: Taxonomy smoothing ───────────────────────────────────────
taxonomy_full = pd.read_csv(BASE / 'taxonomy.csv')
ALPHA_GENUS = 0.15
ALPHA_CLASS = 0.05

genus_groups = {}
class_groups  = {}
for i, label in enumerate(PRIMARY_LABELS):
    row = taxonomy_full[taxonomy_full['primary_label'] == label]
    if len(row) == 0: continue
    row   = row.iloc[0]
    sci   = row.get('scientific_name', '')
    genus = str(sci).split()[0] if (sci and pd.notna(sci)) else 'unknown'
    genus_groups.setdefault(genus, []).append(i)
    cls = str(row.get('class_name', 'unknown'))
    class_groups.setdefault(cls, []).append(i)

multi_genera = {g: v for g, v in genus_groups.items() if len(v) > 1}
print(f"Taxonomy: {len(genus_groups)} genera ({len(multi_genera)} multi), {len(class_groups)} classes")

smoothed = blended.copy()
for genus, idxs in genus_groups.items():
    if len(idxs) > 1:
        g_mean = blended[:, idxs].mean(axis=1, keepdims=True)
        smoothed[:, idxs] = (1-ALPHA_GENUS)*blended[:, idxs] + ALPHA_GENUS*g_mean
for cls, idxs in class_groups.items():
    if len(idxs) > 1:
        c_mean = smoothed[:, idxs].mean(axis=1, keepdims=True)
        smoothed[:, idxs] = (1-ALPHA_CLASS)*smoothed[:, idxs] + ALPHA_CLASS*c_mean
blended = smoothed

# ── Step 3d: Per-class threshold calibration (from 0.949 notebook) ────
# Calibrate thresholds using training soundscape labels as proxy OOF.
# This is what the 0.949 notebook does — rescales probabilities so that
# species with consistently high/low predictions are re-centred at 0.5.
# NO new data required — uses train_soundscapes_labels.csv already loaded.
try:
    from sklearn.isotonic import IsotonicRegression

    # Build training probability proxy from ProtoSSM on training windows
    # We use the Bayesian prior directly as a per-species proxy for
    # expected frequency — remap blended scores through isotonic regression
    # calibrated on global_prior vs per-window label frequency.
    sc_label_calib = pd.read_csv(BASE / 'train_soundscapes_labels.csv')
    sc_label_calib['end_sec'] = pd.to_timedelta(
        sc_label_calib['end']).dt.total_seconds().astype(int)
    sc_label_calib['row_id'] = (
        sc_label_calib['filename'].str.replace('.ogg','',regex=False)
        + '_' + sc_label_calib['end_sec'].astype(str))

    # Build per-window label matrix for training soundscapes
    n_tr_wins = len(sc_label_calib['row_id'].unique())
    tr_rids   = sorted(sc_label_calib['row_id'].unique())
    rid2pos   = {r: i for i, r in enumerate(tr_rids)}
    Y_tr_calib = np.zeros((len(tr_rids), N_CLASSES), dtype=np.float32)
    for _, row in sc_label_calib.iterrows():
        pos = rid2pos.get(row['row_id'])
        if pos is None: continue
        for lbl in str(row['primary_label']).split(';'):
            lbl = lbl.strip()
            if lbl in label_to_idx:
                Y_tr_calib[pos, label_to_idx[lbl]] = 1.0

    # Per-class threshold calibration via isotonic regression
    # Use global_prior as the proxy "predicted probability" for training windows
    PER_CLASS_THRESHOLDS = np.full(N_CLASSES, 0.5, dtype=np.float32)
    n_calibrated = 0
    threshold_grid = (
        [round(t, 3) for t in np.arange(0.20, 0.45, 0.025)]
        + [round(t, 3) for t in np.arange(0.45, 0.75, 0.05)]
    )

    for c in range(N_CLASSES):
        y_true = Y_tr_calib[:, c]
        if y_true.sum() < 3:
            continue
        # Use global_prior[c] as a flat predicted probability for each window
        # Isotonic regression maps this to calibrated probabilities
        y_prob = np.full(len(tr_rids), float(global_prior[c]), dtype=np.float32)
        y_prob += np.random.RandomState(c).normal(0, 0.01, len(tr_rids)).astype(np.float32)
        y_prob = np.clip(y_prob, 0, 1)
        try:
            ir = IsotonicRegression(out_of_bounds='clip')
            ir.fit(y_prob, y_true)
            y_cal = ir.transform(y_prob)
        except:
            continue
        best_f1, best_t = 0.0, 0.5
        for t in threshold_grid:
            pred = (y_cal >= t).astype(int)
            tp = ((pred==1) & (y_true==1)).sum()
            fp = ((pred==1) & (y_true==0)).sum()
            fn = ((pred==0) & (y_true==1)).sum()
            prec = tp / (tp + fp + 1e-8)
            rec  = tp / (tp + fn + 1e-8)
            f1   = 2 * prec * rec / (prec + rec + 1e-8)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        PER_CLASS_THRESHOLDS[c] = best_t
        n_calibrated += 1

    # Apply thresholds: rescale probabilities around each per-class threshold
    blended_cal = blended.copy()
    for c in range(N_CLASSES):
        t = PER_CLASS_THRESHOLDS[c]
        above = blended[:, c] > t
        blended_cal[above, c]  = 0.5 + 0.5 * (blended[above, c]  - t) / (1 - t + 1e-8)
        blended_cal[~above, c] = 0.5 * blended[~above, c] / (t + 1e-8)
    blended = np.clip(blended_cal, 0.0, 1.0)
    print(f"Per-class calibration: {n_calibrated} classes calibrated.")
    print(f"  Threshold range: [{PER_CLASS_THRESHOLDS.min():.2f}, {PER_CLASS_THRESHOLDS.max():.2f}]")
    print(f"  Mean threshold: {PER_CLASS_THRESHOLDS.mean():.3f}")

except Exception as e:
    print(f"Per-class calibration skipped: {e}")

# ── Step 4: Build submission ──────────────────────────────────────────
IS_DRY_RUN = len(test_paths) == 0

if IS_DRY_RUN:
    print("DRY RUN: No test files found")
    sample_sub_df = pd.read_csv(BASE / 'sample_submission.csv')
    sub = sample_sub_df.copy()
    for i, label in enumerate(PRIMARY_LABELS):
        sub[label] = float(blended[:, i].mean())
else:
    sub = pd.DataFrame(blended, columns=PRIMARY_LABELS)
    sub.insert(0, 'row_id', meta_te['row_id'].values)

sub.to_csv('submission.csv', index=False)
print(f"submission.csv saved: {sub.shape}")
print(f"IS_DRY_RUN: {IS_DRY_RUN}")
print(f"Files processed: {len(test_paths)}")
print(f"Total wall time: {time.time()-t0:.1f}s")

Running TaxaMoE inference...
TaxaMoE probs: (240, 234)

Running ProtoSSM v4 inference...
ProtoSSM v4 probs: (240, 234)
Applying logit-space Bayesian prior (lambda=0.625)...
  Proto adj range: [0.0000, 0.9983]
Proto probs after scaling: [0.0000, 0.9959]
Perch direct: (240, 234), nonzero: 229

Running SED inference...
Loaded 5 SED folds
SED probs: (240, 234)

Blending...
4-component blend: SED 42% / ProtoV4 30% / Perch 8% / BiGRU 20%
Gates 1-3 applied.
Sonotype mirroring: 10 columns.
Rare-class suppression: 44 species.
Taxonomy: 161 genera (29 multi), 5 classes
Per-class calibration: 64 classes calibrated.
  Threshold range: [0.20, 0.50]
  Mean threshold: 0.477
submission.csv saved: (240, 235)
IS_DRY_RUN: False
Files processed: 20
Total wall time: 70.1s


In [11]:
tax_check = pd.read_csv(BASE / 'taxonomy.csv')
print("Columns:", tax_check.columns.tolist())

# Only reference columns that exist
print("\nClass null count:", tax_check['class_name'].isna().sum(), "of", len(tax_check))
print("\nSample class values:", tax_check['class_name'].dropna().unique()[:10])
print("Sample scientific names:", tax_check['scientific_name'].dropna().unique()[:10])
print("\nClass value counts (top 10):", tax_check['class_name'].value_counts().head(10).to_dict())

Columns: ['primary_label', 'inat_taxon_id', 'scientific_name', 'common_name', 'class_name']

Class null count: 0 of 234

Sample class values: ['Insecta' 'Reptilia' 'Amphibia' 'Mammalia' 'Aves']
Sample scientific names: ['Guyalna cuta' 'Caiman yacare' 'Leptodactylus luctator'
 'Adenomera guarani' 'Lysapsus limellum' 'Equus caballus'
 'Leptodactylus syphax' 'Leptodactylus mystacinus'
 'Leptodactylus podicipinus' 'Leptodactylus elenae']

Class value counts (top 10): {'Aves': 162, 'Amphibia': 35, 'Insecta': 28, 'Mammalia': 8, 'Reptilia': 1}


In [12]:
import time

# Time Perch alone
t0 = time.time()
emb_test2, logits_test2, meta_test2 = run_perch_batch(test_paths[:5])
perch_time = time.time() - t0
print(f"Perch: {perch_time:.1f}s for 5 files → {perch_time/5*600/60:.1f} min for 600")

# Time SED alone
t0 = time.time()
for path in test_paths[:5]:
    y, _ = sf.read(str(path), dtype='float32', always_2d=False)
    if y.ndim == 2: y = y.mean(1)
    if len(y) < FILE_SAMPLES: y = np.pad(y,(0,FILE_SAMPLES-len(y)))
    chunks = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
    mel = audio_to_mel(chunks)
    for sess_sed in sed_sessions:
        outs = sess_sed.run(None, {sess_sed.get_inputs()[0].name: mel})
sed_time = time.time() - t0
print(f"SED:   {sed_time:.1f}s for 5 files → {sed_time/5*600/60:.1f} min for 600")

Perch: 21.4s for 5 files → 42.9 min for 600
SED:   11.6s for 5 files → 23.2 min for 600


In [13]:
sub_check = pd.read_csv('submission.csv')
print(f"Shape:        {sub_check.shape}")
print(f"Columns[0]:   {sub_check.columns[0]}")
print(f"Columns[1:3]: {sub_check.columns[1:3].tolist()}")
print(f"NaN count:    {sub_check.isna().sum().sum()}")
print(f"Min value:    {sub_check.iloc[:,1:].min().min():.4f}")
print(f"Max value:    {sub_check.iloc[:,1:].max().max():.4f}")
print(f"\nFirst row_id: {sub_check['row_id'].iloc[0]}")
print(f"Last row_id:  {sub_check['row_id'].iloc[-1]}")

Shape:        (240, 235)
Columns[0]:   row_id
Columns[1:3]: ['1161364', '116570']
NaN count:    0
Min value:    0.0281
Max value:    0.9899

First row_id: BC2026_Train_0001_S08_20250606_030007_5
Last row_id:  BC2026_Train_0020_S22_20211104_231500_60


In [14]:
print("lproto_probs:", lproto_probs is None if 'lproto_probs' not in dir() else lproto_probs)

lproto_probs: [[0.03695598 0.57411116 0.03840379 ... 0.02413842 0.05479923 0.08582102]
 [0.0366779  0.56932825 0.03321163 ... 0.02077467 0.05347601 0.0676202 ]
 [0.03572678 0.5646645  0.03307831 ... 0.02059413 0.04793788 0.04867855]
 ...
 [0.00629528 0.08138756 0.00627165 ... 0.01338442 0.01114163 0.04346408]
 [0.00686056 0.08195516 0.0076275  ... 0.01380141 0.01148243 0.03061301]
 [0.00583005 0.08521138 0.00851787 ... 0.01892979 0.0141533  0.03275516]]
